In [1]:
# =================================================================================================
# NHANES 2011-2012 DIETARY MERGE - FIDELITY PRESERVING
#
# Source:
#
#   DR1TOT_G_corrected.csv
#   DR2TOT_G_corrected.csv
#
#
# Purpose:
#
#   Merge corrected NHANES Dietary Day 1 and Day 2 Total Nutrient
#   Intake files into one participant-level Dietary CSV.
#
#
# Final organization:
#
#   SEQN
#
#   WTDRD1
#   WTDR2D
#   DRABF
#   DRDINT
#
#   [Day 1 unique variables]
#
#   [Day 2 unique variables]
#
#
# Fidelity rules:
#
#   1. Preserve Dietary decimal strings AS IS.
#
#   2. DO NOT round Dietary measurements.
#
#   3. DO NOT use float_format=.
#
#   4. DO NOT convert nutrient fields to integers.
#
#   5. Read already-corrected CSVs using dtype=str.
#
#   6. Validate shared Day 1 / Day 2 fields before deduplicating.
#
#   7. Keep one canonical copy of:
#
#          WTDRD1
#          WTDR2D
#          DRABF
#          DRDINT
#
#   8. Prevent engineered fields such as:
#
#          WTDRD1_DAY_2_TOTAL_INTAKE
#          WTDR2D_DAY_2_TOTAL_INTAKE
#          DRABF_DAY_2_TOTAL_INTAKE
#          DRDINT_DAY_2_TOTAL_INTAKE
#
#   9. Every participant-level merge must be one-to-one.
#
#  10. Verify exact text fidelity before and after CSV save.
#
#
# CDC 2011-2012 benchmarks:
#
#   DR1TOT_G rows             = 9,338
#   DR2TOT_G rows             = 9,338
#
#   WTDRD1 populated          = 8,519
#   WTDR2D populated          = 7,605
#
#   DRDINT:
#
#       1 = Day 1 only        =   914
#       2 = Day 1 + Day 2     = 7,605
#       missing               =   819
#
#   DRABF:
#
#       1 = Yes               =   130
#       2 = No                = 8,389
#       missing               =   819
#
# =================================================================================================


# =================================================================================================
# 1. IMPORTS
# =================================================================================================

import pandas as pd
import numpy as np
from pathlib import Path


# =================================================================================================
# 2. PATHS
# =================================================================================================

BASE_DIR = Path().resolve().parent


DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Dietary"
)


DAY1_FILE = (
    DATA_DIR
    / "DR1TOT_G_corrected.csv"
)


DAY2_FILE = (
    DATA_DIR
    / "DR2TOT_G_corrected.csv"
)


OUTPUT_DIR = (
    BASE_DIR
    / "data"
    / "processed_v3"
    / "2011_2012"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


OUTPUT_PATH = (
    OUTPUT_DIR
    / "neighbourhood_nhanes_dietary_2011_2012_raw_corrected.csv"
)


# =================================================================================================
# 3. READ CORRECTED CSV FILES AS TEXT
# =================================================================================================

#
# CRITICAL:
#
# dtype=str ensures that already-corrected Dietary decimal
# representations remain unchanged during this MERGE step.
#
#
# Example:
#
#     0.028999999999
#
# remains:
#
#     0.028999999999
#
#
# We are NOT recalculating any nutrient value.
#

day1 = pd.read_csv(
    DAY1_FILE,
    dtype=str,
    keep_default_na=False
)


day2 = pd.read_csv(
    DAY2_FILE,
    dtype=str,
    keep_default_na=False
)


print("=" * 110)

print(
    "NHANES 2011-2012 DIETARY DAY 1 + DAY 2 MERGE"
)

print("=" * 110)


print(
    "DR1TOT_G shape:",
    day1.shape
)


print(
    "DR2TOT_G shape:",
    day2.shape
)


# =================================================================================================
# 4. STANDARDIZE COLUMN NAMES
# =================================================================================================

day1.columns = (
    day1.columns
    .str.strip()
    .str.upper()
)


day2.columns = (
    day2.columns
    .str.strip()
    .str.upper()
)


# =================================================================================================
# 5. CYCLE-SPECIFIC CDC BENCHMARKS
# =================================================================================================

EXPECTED_ROWS = 9338


EXPECTED_DAY1_WEIGHT_NONMISSING = 8519


EXPECTED_DAY2_WEIGHT_NONMISSING = 7605


EXPECTED_DRDINT = {

    "1": 914,

    "2": 7605,

    "": 819

}


EXPECTED_DRABF = {

    "1": 130,

    "2": 8389,

    "": 819

}


# =================================================================================================
# 6. VERIFY SEQN EXISTS
# =================================================================================================

for name, dataframe in [

    ("DR1TOT_G", day1),

    ("DR2TOT_G", day2)

]:

    if "SEQN" not in dataframe.columns:

        raise ValueError(
            f"{name} does not contain SEQN."
        )


print(
    "PASS: SEQN exists in both files."
)


# =================================================================================================
# 7. VERIFY OFFICIAL ROW COUNTS
# =================================================================================================

if len(day1) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR1TOT_G row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day1):,}"
    )


if len(day2) != EXPECTED_ROWS:

    raise ValueError(
        "Unexpected DR2TOT_G row count.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(day2):,}"
    )


print(
    "PASS: Both DR1TOT_G and DR2TOT_G "
    "contain 9,338 participant rows."
)


# =================================================================================================
# 8. SEQN VALIDATION FUNCTION
# =================================================================================================

def validate_seqn(
    dataframe,
    name
):

    print(
        f"\n--- SEQN CHECK: {name} ---"
    )


    blank_count = int(

        dataframe["SEQN"]
        .eq("")
        .sum()

    )


    duplicate_count = int(

        dataframe["SEQN"]
        .duplicated()
        .sum()

    )


    dot_zero_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            dataframe["SEQN"]
            .str.endswith(".0")
        )

        .sum()

    )


    invalid_format_count = int(

        (
            dataframe["SEQN"].ne("")
            &
            ~dataframe["SEQN"]
            .str.fullmatch(r"\d+")
        )

        .sum()

    )


    print(
        "Rows:",
        f"{len(dataframe):,}"
    )


    print(
        "Unique SEQN:",
        f"{dataframe['SEQN'].nunique():,}"
    )


    print(
        "Blank SEQN:",
        blank_count
    )


    print(
        "Duplicate SEQN:",
        duplicate_count
    )


    print(
        "SEQN ending '.0':",
        dot_zero_count
    )


    print(
        "Invalid SEQN format:",
        invalid_format_count
    )


    if blank_count != 0:

        raise ValueError(
            f"{name}: blank SEQN found."
        )


    if duplicate_count != 0:

        raise ValueError(
            f"{name}: duplicate SEQN found."
        )


    if dot_zero_count != 0:

        raise ValueError(
            f"{name}: SEQN still contains '.0'."
        )


    if invalid_format_count != 0:

        raise ValueError(
            f"{name}: SEQN contains invalid formatting."
        )


    print(
        "PASS"
    )


# =================================================================================================
# 9. VALIDATE SEQN IN BOTH FILES
# =================================================================================================

validate_seqn(
    day1,
    "DR1TOT_G"
)


validate_seqn(
    day2,
    "DR2TOT_G"
)


print(
    "\nPASS: SEQN integrity validated in both files."
)


# =================================================================================================
# 10. VERIFY SAME PARTICIPANT UNIVERSE
# =================================================================================================

day1_seqn = set(
    day1["SEQN"]
)


day2_seqn = set(
    day2["SEQN"]
)


only_day1 = (
    day1_seqn
    -
    day2_seqn
)


only_day2 = (
    day2_seqn
    -
    day1_seqn
)


print(
    "\n--- PARTICIPANT SET CHECK ---"
)


print(
    "Only in DR1TOT_G:",
    len(only_day1)
)


print(
    "Only in DR2TOT_G:",
    len(only_day2)
)


if only_day1 or only_day2:

    print(
        "\nExample Day 1-only SEQN:"
    )

    print(
        sorted(
            only_day1
        )[:20]
    )


    print(
        "\nExample Day 2-only SEQN:"
    )

    print(
        sorted(
            only_day2
        )[:20]
    )


    raise ValueError(
        "DR1TOT_G and DR2TOT_G do not contain "
        "the same participant universe."
    )


print(
    "PASS: Day 1 and Day 2 contain the same "
    "9,338 participant SEQN."
)


# =================================================================================================
# 11. FIND VARIABLES WITH THE SAME NAME IN BOTH FILES
# =================================================================================================

shared_columns_detected = [

    col

    for col in day1.columns

    if (
        col in day2.columns
        and
        col != "SEQN"
    )

]


print(
    "\n--- SHARED VARIABLES DETECTED ---"
)


print(
    shared_columns_detected
)


# =================================================================================================
# 12. OFFICIAL EXPECTED SHARED VARIABLES
# =================================================================================================

#
# CDC explicitly identifies these variables as having
# the same name in both Day 1 and Day 2 files.
#

SHARED_COLUMNS = [

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


if set(
    shared_columns_detected
) != set(
    SHARED_COLUMNS
):

    raise ValueError(
        "Unexpected shared-column pattern.\n\n"
        f"Expected:\n{SHARED_COLUMNS}\n\n"
        f"Detected:\n{shared_columns_detected}"
    )


print(
    "PASS: Exactly four expected shared "
    "Dietary variables found."
)


# =================================================================================================
# 13. BUILD SHARED-VARIABLE COMPARISON
# =================================================================================================

shared_comparison = (

    day1[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="outer",

        suffixes=(
            "_DAY1",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


# =================================================================================================
# 14. VALIDATE SHARED VARIABLES NUMERICALLY
# =================================================================================================

#
# IMPORTANT:
#
# We compare NUMERIC COPIES only for validation.
#
# The actual source strings remain untouched.
#
#
# This prevents a false failure if:
#
#     "1"
#
# and:
#
#     "1.0"
#
# represent the same source value.
#

print(
    "\n--- SHARED VARIABLE FIDELITY CHECK ---"
)


for col in SHARED_COLUMNS:

    day1_raw = (
        shared_comparison[
            f"{col}_DAY1"
        ]
    )


    day2_raw = (
        shared_comparison[
            f"{col}_DAY2"
        ]
    )


    day1_missing = (
        day1_raw.eq("")
    )


    day2_missing = (
        day2_raw.eq("")
    )


    missing_mismatch_count = int(

        (
            day1_missing
            !=
            day2_missing
        )

        .sum()

    )


    both_populated = (
        ~day1_missing
        &
        ~day2_missing
    )


    day1_numeric = pd.to_numeric(

        day1_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    day2_numeric = pd.to_numeric(

        day2_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_equal = np.isclose(

        day1_numeric.to_numpy(
            dtype=float
        ),

        day2_numeric.to_numpy(
            dtype=float
        ),

        rtol=0,

        atol=0

    )


    numeric_mismatch_count = int(

        (
            ~numeric_equal
        )

        .sum()

    )


    #
    # Separate raw-text diagnostic.
    #
    # Text differences do NOT alter source values.
    #

    day1_text = (

        day1_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    day2_text = (

        day2_raw.loc[
            both_populated
        ]

        .reset_index(
            drop=True
        )

    )


    text_difference_count = int(

        (
            day1_text
            !=
            day2_text
        )

        .sum()

    )


    print(
        f"\n{col}"
    )


    print(
        "  Both populated:",
        f"{int(both_populated.sum()):,}"
    )


    print(
        "  Missing-position mismatches:",
        f"{missing_mismatch_count:,}"
    )


    print(
        "  True numeric mismatches:",
        f"{numeric_mismatch_count:,}"
    )


    print(
        "  Raw text-format differences:",
        f"{text_difference_count:,}"
    )


    if missing_mismatch_count != 0:

        raise ValueError(
            f"{col}: missing positions differ "
            "between DR1TOT_G and DR2TOT_G."
        )


    if numeric_mismatch_count != 0:

        comparison_rows = (

            shared_comparison.loc[
                both_populated
            ]

            .reset_index(
                drop=True
            )

        )


        comparison_rows[
            "NUMERIC_MATCH"
        ] = numeric_equal


        print(
            "\nTRUE NUMERIC MISMATCH EXAMPLES:"
        )


        display(

            comparison_rows.loc[
                ~comparison_rows[
                    "NUMERIC_MATCH"
                ],
                [
                    "SEQN",
                    f"{col}_DAY1",
                    f"{col}_DAY2"
                ]
            ]

            .head(20)

        )


        raise ValueError(
            f"{col}: true numeric values differ "
            "between DR1TOT_G and DR2TOT_G."
        )


print(
    "\nPASS: WTDRD1, WTDR2D, DRABF and DRDINT "
    "agree between Day 1 and Day 2."
)


# =================================================================================================
# 15. CDC WTDRD1 / WTDR2D COUNTS
# =================================================================================================

day1_weight_count = int(

    day1["WTDRD1"]
    .ne("")
    .sum()

)


day2_weight_count = int(

    day1["WTDR2D"]
    .ne("")
    .sum()

)


print(
    "\n--- CDC DIETARY SAMPLE WEIGHT CHECK ---"
)


print(
    "WTDRD1 populated:",
    f"{day1_weight_count:,}"
)


print(
    "WTDR2D populated:",
    f"{day2_weight_count:,}"
)


if (
    day1_weight_count
    !=
    EXPECTED_DAY1_WEIGHT_NONMISSING
):

    raise ValueError(
        "WTDRD1 count differs from CDC.\n"
        f"Expected: {EXPECTED_DAY1_WEIGHT_NONMISSING:,}\n"
        f"Actual:   {day1_weight_count:,}"
    )


if (
    day2_weight_count
    !=
    EXPECTED_DAY2_WEIGHT_NONMISSING
):

    raise ValueError(
        "WTDR2D count differs from CDC.\n"
        f"Expected: {EXPECTED_DAY2_WEIGHT_NONMISSING:,}\n"
        f"Actual:   {day2_weight_count:,}"
    )


print(
    "PASS: Dietary weight counts match CDC."
)


# =================================================================================================
# 16. CDC DRDINT CHECK
# =================================================================================================

drdint_counts = (

    day1["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRDINT CHECK ---"
)


print(
    drdint_counts
)


if drdint_counts != EXPECTED_DRDINT:

    raise ValueError(
        "DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{drdint_counts}"
    )


print(
    "PASS: DRDINT matches CDC."
)


# =================================================================================================
# 17. CDC DRABF CHECK
# =================================================================================================

drabf_counts = (

    day1["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\n--- CDC DRABF CHECK ---"
)


print(
    drabf_counts
)


if drabf_counts != EXPECTED_DRABF:

    raise ValueError(
        "DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{drabf_counts}"
    )


print(
    "PASS: DRABF matches CDC."
)


# =================================================================================================
# 18. SAVE ORIGINAL COLUMN ORDERS
# =================================================================================================

original_day1_columns = (
    day1.columns.tolist()
)


original_day2_columns = (
    day2.columns.tolist()
)


# =================================================================================================
# 19. REMOVE ONLY REDUNDANT DAY 2 SHARED COPIES
# =================================================================================================

#
# Canonical copies are retained from DR1TOT_G.
#
# The equivalent copies from DR2TOT_G are removed
# BEFORE the merge.
#

day2_for_merge = (

    day2.drop(
        columns=SHARED_COLUMNS
    )

)


print(
    "\nRemoved redundant DR2TOT_G copies:"
)


for col in SHARED_COLUMNS:

    print(
        " -",
        col
    )


# =================================================================================================
# 20. VERIFY NO UNEXPECTED COLUMN COLLISIONS
# =================================================================================================

remaining_overlap = sorted(

    (
        set(
            day1.columns
        )

        &

        set(
            day2_for_merge.columns
        )
    )

    -

    {
        "SEQN"
    }

)


if remaining_overlap:

    raise ValueError(
        "Unexpected overlapping variables remain "
        "before Dietary merge:\n"
        + str(
            remaining_overlap
        )
    )


print(
    "PASS: No unexpected column collisions remain."
)


# =================================================================================================
# 21. MERGE DAY 1 + DAY 2
# =================================================================================================

dietary = (

    day1.merge(

        day2_for_merge,

        on="SEQN",

        how="outer",

        validate="one_to_one"

    )

)


#
# Outer joins can introduce NaN.
#
# Restore them to blank text so the merged representation
# remains consistent with the corrected source CSVs.
#

dietary = (
    dietary.fillna("")
)


print(
    "\nPASS: DR1TOT_G + DR2TOT_G merge completed."
)


# =================================================================================================
# 22. FINAL PARTICIPANT COUNT CHECK
# =================================================================================================

if len(dietary) != EXPECTED_ROWS:

    raise ValueError(
        "Merged Dietary row count changed.\n"
        f"Expected: {EXPECTED_ROWS:,}\n"
        f"Actual:   {len(dietary):,}"
    )


if (
    dietary["SEQN"]
    .nunique()
    !=
    EXPECTED_ROWS
):

    raise ValueError(
        "Merged Dietary unique SEQN count changed."
    )


if (
    dietary["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Merged Dietary dataset contains duplicate SEQN."
    )


if (
    dietary["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Merged Dietary dataset contains blank SEQN."
    )


if (
    dietary["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Merged Dietary dataset contains SEQN '.0'."
    )


print(
    "PASS: Final Dietary dataset contains "
    "9,338 unique participant SEQN."
)


# =================================================================================================
# 23. DUPLICATE COLUMN NAME CHECK
# =================================================================================================

duplicate_column_names = (

    dietary.columns[
        dietary.columns.duplicated()
    ]

    .tolist()

)


if duplicate_column_names:

    raise ValueError(
        "Duplicate column names found:\n"
        + str(
            duplicate_column_names
        )
    )


print(
    "PASS: No duplicate column names."
)


# =================================================================================================
# 24. ENGINEERED DUPLICATE FIELD CHECK
# =================================================================================================

BAD_DUPLICATE_COLUMNS = [

    "WTDRD1_DAY_2_TOTAL_INTAKE",

    "WTDR2D_DAY_2_TOTAL_INTAKE",

    "DRABF_DAY_2_TOTAL_INTAKE",

    "DRDINT_DAY_2_TOTAL_INTAKE",

    "WTDRD1_X",

    "WTDRD1_Y",

    "WTDR2D_X",

    "WTDR2D_Y",

    "DRABF_X",

    "DRABF_Y",

    "DRDINT_X",

    "DRDINT_Y"

]


existing_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in dietary.columns

]


if existing_bad_columns:

    raise ValueError(
        "Engineered Dietary duplicate columns remain:\n"
        + str(
            existing_bad_columns
        )
    )


print(
    "PASS: No engineered Dietary duplicate fields."
)


# =================================================================================================
# 25. CREATE FINAL LOGICAL COLUMN ORDER
# =================================================================================================

SHARED_ORDER = [

    "SEQN",

    "WTDRD1",

    "WTDR2D",

    "DRABF",

    "DRDINT"

]


DAY1_UNIQUE_COLUMNS = [

    col

    for col in original_day1_columns

    if col not in SHARED_ORDER

]


DAY2_UNIQUE_COLUMNS = [

    col

    for col in original_day2_columns

    if (
        col != "SEQN"
        and
        col not in SHARED_COLUMNS
    )

]


FINAL_COLUMN_ORDER = (

    SHARED_ORDER

    +

    DAY1_UNIQUE_COLUMNS

    +

    DAY2_UNIQUE_COLUMNS

)


# =================================================================================================
# 26. VALIDATE FINAL COLUMN ORDER
# =================================================================================================

if (
    len(FINAL_COLUMN_ORDER)
    !=
    len(set(FINAL_COLUMN_ORDER))
):

    raise ValueError(
        "FINAL_COLUMN_ORDER contains duplicate names."
    )


missing_from_order = (

    set(
        dietary.columns
    )

    -

    set(
        FINAL_COLUMN_ORDER
    )

)


extra_in_order = (

    set(
        FINAL_COLUMN_ORDER
    )

    -

    set(
        dietary.columns
    )

)


if missing_from_order or extra_in_order:

    raise ValueError(
        "Final column order does not match merged dataframe.\n\n"
        f"Missing from order:\n"
        f"{sorted(missing_from_order)}\n\n"
        f"Extra in order:\n"
        f"{sorted(extra_in_order)}"
    )


dietary = (
    dietary[
        FINAL_COLUMN_ORDER
    ]
)


print(
    "PASS: Final column order created."
)


# =================================================================================================
# 27. DISPLAY FINAL STRUCTURE
# =================================================================================================

print(
    "\n--- FINAL DIETARY STRUCTURE ---"
)


print(
    "Shared fields:",
    len(SHARED_ORDER)
)


print(
    "Day 1 unique fields:",
    len(DAY1_UNIQUE_COLUMNS)
)


print(
    "Day 2 unique fields:",
    len(DAY2_UNIQUE_COLUMNS)
)


print(
    "Total fields:",
    len(dietary.columns)
)


# =================================================================================================
# 28. DAY 1 SOURCE-TO-MERGED EXACT TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DR1TOT_G SOURCE FIDELITY ---"
)


day1_check = (

    day1.merge(

        dietary[
            original_day1_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day1_failures = []


for col in original_day1_columns:

    if col == "SEQN":

        continue


    source_values = (
        day1_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day1_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day1_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day1_failures:

    raise ValueError(
        "DR1TOT_G values changed during merge:\n"
        + str(
            day1_failures
        )
    )


print(
    "PASS: Every retained DR1TOT_G value "
    "preserved text-for-text."
)


# =================================================================================================
# 29. DAY 2 UNIQUE FIELD EXACT TEXT FIDELITY
# =================================================================================================

print(
    "\n--- DR2TOT_G SOURCE FIDELITY ---"
)


day2_check_columns = (

    ["SEQN"]

    +

    DAY2_UNIQUE_COLUMNS

)


day2_check = (

    day2[
        day2_check_columns
    ]

    .merge(

        dietary[
            day2_check_columns
        ],

        on="SEQN",

        how="left",

        suffixes=(
            "_SOURCE",
            "_MERGED"
        ),

        validate="one_to_one"

    )

)


day2_failures = []


for col in DAY2_UNIQUE_COLUMNS:

    source_values = (
        day2_check[
            f"{col}_SOURCE"
        ]
        .fillna("")
    )


    merged_values = (
        day2_check[
            f"{col}_MERGED"
        ]
        .fillna("")
    )


    mismatch_count = int(

        (
            source_values
            !=
            merged_values
        )

        .sum()

    )


    if mismatch_count > 0:

        day2_failures.append(
            (
                col,
                mismatch_count
            )
        )


if day2_failures:

    raise ValueError(
        "DR2TOT_G values changed during merge:\n"
        + str(
            day2_failures
        )
    )


print(
    "PASS: Every unique DR2TOT_G value "
    "preserved text-for-text."
)


# =================================================================================================
# 30. FINAL SHARED-VARIABLE CHECK AGAINST DR2TOT_G
# =================================================================================================

print(
    "\n--- FINAL SHARED FIELD CHECK ---"
)


final_shared_check = (

    dietary[
        ["SEQN"] + SHARED_COLUMNS
    ]

    .merge(

        day2[
            ["SEQN"] + SHARED_COLUMNS
        ],

        on="SEQN",

        how="inner",

        suffixes=(
            "_FINAL",
            "_DAY2"
        ),

        validate="one_to_one"

    )

)


for col in SHARED_COLUMNS:

    final_raw = (
        final_shared_check[
            f"{col}_FINAL"
        ]
    )


    day2_raw = (
        final_shared_check[
            f"{col}_DAY2"
        ]
    )


    final_missing = (
        final_raw.eq("")
    )


    day2_missing = (
        day2_raw.eq("")
    )


    missing_mismatch = int(

        (
            final_missing
            !=
            day2_missing
        )

        .sum()

    )


    both_populated = (
        ~final_missing
        &
        ~day2_missing
    )


    final_numeric = pd.to_numeric(

        final_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    day2_numeric = pd.to_numeric(

        day2_raw.loc[
            both_populated
        ],

        errors="raise"

    )


    numeric_mismatch = int(

        (
            ~np.isclose(

                final_numeric.to_numpy(
                    dtype=float
                ),

                day2_numeric.to_numpy(
                    dtype=float
                ),

                rtol=0,

                atol=0

            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"missing mismatches={missing_mismatch:,}, "
        f"numeric mismatches={numeric_mismatch:,}"
    )


    if missing_mismatch != 0:

        raise ValueError(
            f"{col}: final missing positions differ "
            "from DR2TOT_G."
        )


    if numeric_mismatch != 0:

        raise ValueError(
            f"{col}: final values differ "
            "from DR2TOT_G."
        )


print(
    "PASS: Shared Dietary fields remain "
    "equivalent to both source files."
)


# =================================================================================================
# 31. DIETARY DECIMAL PRESERVATION DIAGNOSTIC
# =================================================================================================

#
# IMPORTANT:
#
# No value conversion occurs here.
#
# These checks simply show that real Dietary decimal
# tokens remain present.
#

REPRESENTATIVE_DECIMAL_FIELDS = [

    "WTDRD1",

    "WTDR2D",

    "DR1TPROT",

    "DR1TCARB",

    "DR1TP204",

    "DR1_320Z",

    "DR2TPROT",

    "DR2TCARB",

    "DR2TP204",

    "DR2_320Z"

]


print(
    "\n--- DIETARY DECIMAL PRESERVATION ---"
)


for col in REPRESENTATIVE_DECIMAL_FIELDS:

    if col not in dietary.columns:

        continue


    populated_count = int(

        dietary[col]
        .ne("")
        .sum()

    )


    decimal_token_count = int(

        (
            dietary[col].ne("")
            &
            dietary[col]
            .str.contains(
                ".",
                regex=False
            )
        )

        .sum()

    )


    print(
        f"{col}: "
        f"populated={populated_count:,}, "
        f"decimal tokens={decimal_token_count:,}"
    )


print(
    "PASS: Dietary values remained text; "
    "no rounding or integer conversion occurred."
)


# =================================================================================================
# 32. KNOWN XPORT TINY-ARTIFACT TOKEN CHECK
# =================================================================================================

TINY_TOKEN = (
    "5.397605346934028e-79"
)


tiny_found = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]
        .str.lower()
        .eq(
            TINY_TOKEN.lower()
        )
        .sum()

    )


    if count > 0:

        tiny_found[
            col
        ] = count


if tiny_found:

    raise ValueError(
        "Known XPORT tiny-zero artifacts remain "
        "in the corrected Dietary inputs:\n"
        + str(
            tiny_found
        )
    )


print(
    "PASS: No known tiny XPORT artifacts."
)


# =================================================================================================
# 33. SCIENTIFIC NOTATION DIAGNOSTIC
# =================================================================================================

#
# This is diagnostic only.
#
# We DO NOT alter or reject a legitimate source token merely because
# it uses scientific notation.
#
# Exact text fidelity is the controlling validation.
#

scientific_notation_fields = {}


for col in dietary.columns:

    if col == "SEQN":

        continue


    count = int(

        dietary[col]

        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )

        .sum()

    )


    if count > 0:

        scientific_notation_fields[
            col
        ] = count


print(
    "\nScientific-notation source tokens:",
    sum(
        scientific_notation_fields.values()
    )
)


if scientific_notation_fields:

    print(
        "INFO: Scientific notation exists in corrected source text."
    )

    print(
        "These values will be preserved AS IS:"
    )

    print(
        scientific_notation_fields
    )


# =================================================================================================
# 34. SAVE FINAL DIETARY CSV
# =================================================================================================

#
# CRITICAL:
#
# No:
#
#     float_format=
#
# No rounding.
#
# No numeric conversion.
#
# Already-corrected Dietary text is written AS IS.
#

dietary.to_csv(

    OUTPUT_PATH,

    index=False

)


print(
    "\nSaved final Dietary file:"
)


print(
    OUTPUT_PATH
)


# =================================================================================================
# 35. RE-READ FINAL FILE AS TEXT
# =================================================================================================

final_df = pd.read_csv(

    OUTPUT_PATH,

    dtype=str,

    keep_default_na=False

)


# =================================================================================================
# 36. POST-SAVE STRUCTURE CHECK
# =================================================================================================

if final_df.shape != dietary.shape:

    raise ValueError(
        "Saved CSV shape changed.\n"
        f"Before save: {dietary.shape}\n"
        f"After save:  {final_df.shape}"
    )


if (
    final_df.columns.tolist()
    !=
    dietary.columns.tolist()
):

    raise ValueError(
        "Saved CSV column order changed."
    )


print(
    "PASS: Saved CSV structure preserved."
)


# =================================================================================================
# 37. EXACT PRE-SAVE / POST-SAVE TEXT FIDELITY
# =================================================================================================

#
# This is the strongest protection for Dietary decimal representation.
#
# Every source token before save must be exactly the same
# after re-reading the final CSV.
#

post_save_failures = []


for col in dietary.columns:

    same = (

        dietary[col]

        .eq(
            final_df[col]
        )

        .all()

    )


    if not same:

        mismatch_count = int(

            (
                dietary[col]
                !=
                final_df[col]
            )

            .sum()

        )


        post_save_failures.append(
            (
                col,
                mismatch_count
            )
        )


if post_save_failures:

    raise ValueError(
        "CSV text changed during export:\n"
        + str(
            post_save_failures
        )
    )


print(
    "PASS: Exact pre-save/post-save text fidelity."
)


# =================================================================================================
# 38. FINAL SEQN CHECK
# =================================================================================================

if (
    final_df["SEQN"]
    .duplicated()
    .any()
):

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    final_df["SEQN"]
    .eq("")
    .any()
):

    raise ValueError(
        "Final CSV contains blank SEQN."
    )


if (
    final_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV contains SEQN '.0'."
    )


if (

    ~final_df["SEQN"]
    .str.fullmatch(r"\d+")

).any():

    raise ValueError(
        "Final CSV contains invalid SEQN formatting."
    )


print(
    "PASS: Final SEQN integrity."
)


# =================================================================================================
# 39. FINAL ENGINEERED DUPLICATE CHECK
# =================================================================================================

final_bad_columns = [

    col

    for col in BAD_DUPLICATE_COLUMNS

    if col in final_df.columns

]


if final_bad_columns:

    raise ValueError(
        "Engineered duplicate fields remain:\n"
        + str(
            final_bad_columns
        )
    )


print(
    "PASS: No redundant Day 2 shared-variable copies."
)


# =================================================================================================
# 40. FINAL SAMPLE WEIGHT COUNTS
# =================================================================================================

final_wtdrd1_count = int(

    final_df["WTDRD1"]
    .ne("")
    .sum()

)


final_wtdr2d_count = int(

    final_df["WTDR2D"]
    .ne("")
    .sum()

)


print(
    "\n--- FINAL CDC WEIGHT CHECK ---"
)


print(
    "WTDRD1:",
    f"{final_wtdrd1_count:,}"
)


print(
    "WTDR2D:",
    f"{final_wtdr2d_count:,}"
)


if final_wtdrd1_count != 8519:

    raise ValueError(
        "Final WTDRD1 count differs from CDC."
    )


if final_wtdr2d_count != 7605:

    raise ValueError(
        "Final WTDR2D count differs from CDC."
    )


print(
    "PASS: Final Dietary sample weights match CDC."
)


# =================================================================================================
# 41. FINAL DRDINT CHECK
# =================================================================================================

final_drdint = (

    final_df["DRDINT"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\nFinal DRDINT:"
)


print(
    final_drdint
)


if final_drdint != EXPECTED_DRDINT:

    raise ValueError(
        "Final DRDINT distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRDINT}\n\n"
        f"Actual:\n{final_drdint}"
    )


print(
    "PASS: Final DRDINT matches CDC."
)


# =================================================================================================
# 42. FINAL DRABF CHECK
# =================================================================================================

final_drabf = (

    final_df["DRABF"]

    .value_counts(
        dropna=False
    )

    .to_dict()

)


print(
    "\nFinal DRABF:"
)


print(
    final_drabf
)


if final_drabf != EXPECTED_DRABF:

    raise ValueError(
        "Final DRABF distribution differs from CDC.\n\n"
        f"Expected:\n{EXPECTED_DRABF}\n\n"
        f"Actual:\n{final_drabf}"
    )


print(
    "PASS: Final DRABF matches CDC."
)


# =================================================================================================
# 43. FINAL TINY-XPORT CHECK
# =================================================================================================

final_tiny_found = {}


for col in final_df.columns:

    if col == "SEQN":

        continue


    count = int(

        final_df[col]

        .str.lower()

        .eq(
            TINY_TOKEN.lower()
        )

        .sum()

    )


    if count > 0:

        final_tiny_found[
            col
        ] = count


if final_tiny_found:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV:\n"
        + str(
            final_tiny_found
        )
    )


print(
    "PASS: No known tiny XPORT artifacts after save."
)


# =================================================================================================
# 44. FINAL SUMMARY
# =================================================================================================

print(
    "\n"
    + "=" * 110
)


print(
    "NHANES 2011-2012 DIETARY MERGE COMPLETE"
)


print(
    "=" * 110
)


print(
    "Cycle:"
)


print(
    "  2011-2012"
)


print(
    ""
)


print(
    "Sources:"
)


print(
    "  DR1TOT_G_corrected.csv"
)


print(
    "  DR2TOT_G_corrected.csv"
)


print(
    ""
)


print(
    "Rows:",
    f"{len(final_df):,}"
)


print(
    "Unique SEQN:",
    f"{final_df['SEQN'].nunique():,}"
)


print(
    "Columns:",
    f"{len(final_df.columns):,}"
)


print(
    ""
)


print(
    "CDC benchmarks:"
)


print(
    "  Total participant rows = 9,338"
)


print(
    "  WTDRD1 populated        = 8,519"
)


print(
    "  WTDR2D populated        = 7,605"
)


print(
    ""
)


print(
    "DRDINT:"
)


print(
    "  Day 1 only       =   914"
)


print(
    "  Day 1 + Day 2    = 7,605"
)


print(
    "  Missing          =   819"
)


print(
    ""
)


print(
    "DRABF:"
)


print(
    "  Yes              =   130"
)


print(
    "  No               = 8,389"
)


print(
    "  Missing          =   819"
)


print(
    ""
)


print(
    "Shared variables retained once:"
)


print(
    "  WTDRD1"
)


print(
    "  WTDR2D"
)


print(
    "  DRABF"
)


print(
    "  DRDINT"
)


print(
    ""
)


print(
    "Final organization:"
)


print(
    "  SEQN"
)


print(
    "  Shared Dietary fields"
)


print(
    "  Day 1 fields"
)


print(
    "  Day 2 fields"
)


print(
    ""
)


print(
    "Duplicate SEQN: 0"
)


print(
    "SEQN '.0': 0"
)


print(
    "Engineered Dietary duplicates: 0"
)


print(
    "Known tiny XPORT artifacts: 0"
)


print(
    "Day 1 retained-value changes: 0"
)


print(
    "Day 2 retained-value changes: 0"
)


print(
    "Dietary decimal reformatting during merge: 0"
)


print(
    "Pre-save/post-save text fidelity: PASS"
)


print(
    "=" * 110
)

NHANES 2011-2012 DIETARY DAY 1 + DAY 2 MERGE
DR1TOT_G shape: (9338, 166)
DR2TOT_G shape: (9338, 83)
PASS: SEQN exists in both files.
PASS: Both DR1TOT_G and DR2TOT_G contain 9,338 participant rows.

--- SEQN CHECK: DR1TOT_G ---
Rows: 9,338
Unique SEQN: 9,338
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

--- SEQN CHECK: DR2TOT_G ---
Rows: 9,338
Unique SEQN: 9,338
Blank SEQN: 0
Duplicate SEQN: 0
SEQN ending '.0': 0
Invalid SEQN format: 0
PASS

PASS: SEQN integrity validated in both files.

--- PARTICIPANT SET CHECK ---
Only in DR1TOT_G: 0
Only in DR2TOT_G: 0
PASS: Day 1 and Day 2 contain the same 9,338 participant SEQN.

--- SHARED VARIABLES DETECTED ---
['WTDRD1', 'WTDR2D', 'DRABF', 'DRDINT']
PASS: Exactly four expected shared Dietary variables found.

--- SHARED VARIABLE FIDELITY CHECK ---

WTDRD1
  Both populated: 8,519
  Missing-position mismatches: 0
  True numeric mismatches: 0
  Raw text-format differences: 0

WTDR2D
  Both populated: 7,605
  Mis